<a href="https://colab.research.google.com/github/100522401/P1_AA_100522396_100522401/blob/main/notebooks/Prediccion_competicion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Predicción sobre el conjunto de competición

**Mateo Zariquiey Pérez 100522396**

**Héctor Fuertes Muñoz 100522401**


En este notebook se carga el modelo final ya entrenado y almacenado el `modelo_final.joblib`, con el objetivo de generar las predicciones sobre el conjunto de competición. El resultado se exportará al fichero `predicciones.csv`

## Importación de librerías y rutas

En este bloque se definen las rutas del modelo final, el dataset de competición y el fichero donde se almacenarán las predicciones.

In [1]:
import numpy as np
import pandas as pd
from joblib import load

MODEL_PATH = 'modelo_final.joblib'
COMP_PATH = '../data/bank_competition.pkl'
PRED_PATH = 'predicciones.csv'

print('MODEL_PATH:', MODEL_PATH)
print('COMP_PATH:', COMP_PATH)
print('PRED_PATH:', PRED_PATH)

MODEL_PATH: modelo_final.joblib
COMP_PATH: ../data/bank_competition.pkl
PRED_PATH: predicciones.csv


## Funciones auxiliares pipeline
En esta sección se vuelven a definir las funciones auxiliares utilizadas en el pipeline. Su presencia es necesaria para que el `joblib` pueda funcionar correctamente.

In [2]:
def clean_none_values(X):
    X = X.copy()
    return X.replace({None: np.nan})

def transform_pdays(X):
    X = X.copy()
    X["contacted_before"] = (X["pdays"] != -1).astype(int)
    X["pdays"] = X["pdays"].replace(-1, np.nan)
    return X


## Carga del modelo final y del dataset de competición

Una vez ya se han definido funciones auxiliares necesarias para reconstruir correctamente la pipeline, se carga el paquete del modelo final y se extrae la pipeline ya entrenada, junto con el dataset de competición sobre el que se deben realizar las predicciones.

En este caso, la pipeline incluye tanto preprocesado como el clasificador, por loq ue las nuevas instancias pueden pasarse directametne al método `predict` sin transformar manualmente los datos.

In [3]:
pack = load(MODEL_PATH)
final_model = pack["pipeline"]
competition_data = pd.read_pickle(COMP_PATH)

print('Dimensiones del conjunto de competición:', competition_data.shape)
display(competition_data.head())

Dimensiones del conjunto de competición: (162, 16)


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome
5553,43,management,married,tertiary,no,78,yes,no,cellular,21,nov,36,1,109,1,other
915,34,housemaid,married,secondary,no,0,yes,no,unknown,30,oct,154,1,-1,0,unknown
7652,54,technician,married,secondary,no,3323,yes,yes,cellular,8,apr,59,3,-1,0,unknown
5065,43,blue-collar,single,primary,no,-399,no,yes,cellular,28,jul,662,3,-1,0,unknown
3338,35,blue-collar,married,secondary,no,262,no,no,cellular,15,mar,427,1,181,3,success


## Generación de predicciones

El modelo final se utiliza para predecir la variable `deposit` en el conjunto de competición. Después, las predicciones se convierten al formato `yes`/`no` y se guardan en el archivo `predicciones.csv`.

In [4]:
competition_pred = final_model.predict(competition_data)
competition_pred_labels = pd.Series(competition_pred).map({1: 'yes', 0: 'no'})

predicciones_df = pd.DataFrame({'deposit': competition_pred_labels})
predicciones_df.to_csv(PRED_PATH, index=False)

print(f'Predicciones guardadas en: {PRED_PATH}')
display(predicciones_df.head())
print(predicciones_df['deposit'].value_counts())

Predicciones guardadas en: predicciones.csv


,deposit
0,no
1,no
2,no
3,yes
4,yes


deposit
no     89
yes    73
Name: count, dtype: int64


## Verificación del número de predicciones generadas

Para evitar posibles errores, se comprueba que el número de predicciones generadas coincida con el número de instancias a predecir.

In [5]:
print('Número de filas del conjunto de competición:', len(competition_data))
print('Número de filas en predicciones.csv:', len(predicciones_df))

assert len(competition_data) == len(predicciones_df)

Número de filas del conjunto de competición: 162
Número de filas en predicciones.csv: 162


## Instancias de comprobación Streamlit

A continuación, para comprobar el correcto funcionamiento de las predicciones hechas en este cuaderno y las realizadas en la interfaz **Streamlit** se definirán dos nuevas intancias de manera manual.

Entonces, se introducirán las nuevas instancias en la pipeline cargada en este notebook y posteriormente se introducirán en la aplicación de Streamlit. Por último se verificará que las predicciones obtenidas coinciden.

Para la creación de estas nuevas instancias se ha optado por la interpretación de los coeficientes obtenidos previamente en la regresión logística. EN concreto, se he diseñado una instancia que favorece a la clase `yes` de `deposit` y otra con valores que favorecen a la clase `no`.

En primer lugar se define la **instancia positiva**.

In [6]:
instancia_positiva = pd.DataFrame([{
    'age': 32,
    'job': 'student',
    'marital': 'single',
    'education': 'secondary',
    'default': 'no',
    'balance': 1500,
    'housing': 'no',
    'loan': 'no',
    'contact': 'cellular',
    'day': 12,
    'month': 'mar',
    'duration': 420,
    'campaign': 1,
    'pdays': 90,
    'previous': 2,
    'poutcome': 'success'
}])

display(instancia_positiva)

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome
0,32,student,single,secondary,no,1500,no,no,cellular,12,mar,420,1,90,2,success


A continuación, la **instancia negativa**

In [7]:
instancia_negativa = pd.DataFrame([{
    'age': 48,
    'job': 'housemaid',
    'marital': 'married',
    'education': 'primary',
    'default': 'no',
    'balance': 50,
    'housing': 'yes',
    'loan': 'yes',
    'contact': 'unknown',
    'day': 20,
    'month': 'jan',
    'duration': 40,
    'campaign': 4,
    'pdays': -1,
    'previous': 0,
    'poutcome': 'failure'
}])

print("Instancia orientada a predicción negativa:")
display(instancia_negativa)

Instancia orientada a predicción negativa:


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome
0,48,housemaid,married,primary,no,50,yes,yes,unknown,20,jan,40,4,-1,0,failure


Una vez se han definido las instancias, se calculan sus predicciones.

In [8]:
nuevas_instancias = pd.concat([instancia_positiva, instancia_negativa], ignore_index=True)

pred_nuevas = final_model.predict(nuevas_instancias)
pred_nuevas_labels = ['yes' if p == 1 else 'no' for p in pred_nuevas]

print(f"Predicción de la instancia positiva: {pred_nuevas_labels[0]}")
print(f"Predicción de la instancia negativa: {pred_nuevas_labels[1]}")

Predicción de la instancia positiva: yes
Predicción de la instancia negativa: no


Estas dos instancias se introducirán manualmente en la aplicación Streamlit, y se harán capturas que demuestren que las predicciones obtenidas coinciden con las calculadas en este notebook.